# Contego MSSP — Gerador de Relatórios no Google Colab

Gera os relatórios mensais de proteção gerenciada (somente leitura, dados reais da API Acronis) — **HTML + apresentação PPTX** por cliente. **Cada cliente tem a sua própria célula** na seção 5: rode a célula do cliente desejado e o relatório (HTML) e o deck (PPTX) dele são gerados e baixados automaticamente.

## Antes de rodar — configure os *Secrets* do Colab (ícone de chave na barra esquerda):
| Nome | Valor |
|---|---|
| `US_CLOUD_ID` / `US_CLOUD_SECRET` | credenciais da API us-cloud |
| `BR02_ID` / `BR02_SECRET` | credenciais da API br02 |

## Como usar
1. Rode as células **1, 2, 3 e 4** uma vez cada (instalar PowerShell, clonar, credenciais, preparar).
2. Vá até a seção **5** e rode a **célula do cliente** que você quer. Cada célula gera e baixa o **HTML + o PPTX** daquele cliente — pode rodar em qualquer ordem e repetir quando quiser.

_Repositório público, apenas código — nenhuma credencial ou dado de cliente fica aqui._

## 1. Instalar o PowerShell 7

In [ ]:
%%bash
if ! command -v pwsh >/dev/null 2>&1; then
  wget -q https://github.com/PowerShell/PowerShell/releases/download/v7.4.6/powershell_7.4.6-1.deb_amd64.deb -O /tmp/ps.deb
  sudo dpkg -i /tmp/ps.deb >/dev/null 2>&1 || sudo apt-get install -f -y >/dev/null 2>&1
fi
pwsh --version

## 2. Clonar o repositório (público — sem token)

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/oness24/acronis_report_tool.git'
DEST = '/content/acronis_report_tool'
if os.path.isdir(DEST):
    subprocess.run(['git','-C',DEST,'pull'], check=False)
else:
    subprocess.run(['git','clone','--depth','1', REPO_URL, DEST], check=True)
print('repo pronto:', os.listdir(DEST + '/automation')[:8])

## 3. Gravar as credenciais da API (a partir dos Secrets — não ficam no repositório)
Opcional: para relatórios **idênticos aos locais** (gating por contrato, cotas contratadas, M365 "/100"), crie também um Secret `CONTRACTS_JSON` com o conteúdo do seu `contracts.json`. Sem ele, os relatórios usam as licenças ativas (a BIOCAL continua combinada entre os dois data centers).

In [ ]:
import json
from google.colab import userdata
DEST = '/content/acronis_report_tool'
secrets = {
  'us-cloud':   {'clientId': userdata.get('US_CLOUD_ID'), 'secret': userdata.get('US_CLOUD_SECRET')},
  'br02-cloud': {'clientId': userdata.get('BR02_ID'),     'secret': userdata.get('BR02_SECRET')},
}
open(DEST + '/automation/config/secrets.json','w').write(json.dumps(secrets, indent=2))
print('secrets.json gravado para:', [k for k,v in secrets.items() if v.get('secret')])

# Opcional: contratos completos via Secret CONTRACTS_JSON (relatorios identicos aos locais)
try:
    cj = userdata.get('CONTRACTS_JSON')
except Exception:
    cj = None
if cj:
    json.loads(cj)  # valida
    open(DEST + '/automation/config/contracts.json','w', encoding='utf-8').write(cj)
    print('contracts.json gravado a partir de CONTRACTS_JSON -> relatorios completos.')
else:
    print('Sem CONTRACTS_JSON: gating por licencas ativas (BIOCAL ainda e combinada).')

## 4. Preparar o gerador (rode UMA vez)
Instala o `python-pptx` e define a função que gera e baixa um cliente. `MONTH` = mês do relatório (`YYYY-MM`); use `''` para o mês anterior automaticamente. Saída por cliente: **HTML + PPTX + deck.pdf** (o relatório PDF é ignorado no Colab; o deck PPTX e o deck PDF Gamma são gerados porque esta célula instala o `python-pptx` e o `chromium-browser`).

In [ ]:
import subprocess, sys, glob, os, time
from google.colab import files
DEST  = '/content/acronis_report_tool'
RUN   = DEST + '/automation/Run-MonthlyReports.ps1'
MONTH = '2026-05'      # mês do relatório (YYYY-MM); '' = mês anterior

# python-pptx: necessário para gerar o deck .pptx
import subprocess as _sp, sys as _sys
_sp.run([_sys.executable, '-m', 'pip', '-q', 'install', 'python-pptx'])

# chromium-browser: necessario para gerar o deck .pdf
_sp.run('apt-get -qq update && apt-get -qq install -y chromium-browser', shell=True)

def run_ps(extra, echo=True):
    args = ['pwsh','-File', RUN] + (['-Month', MONTH] if MONTH else []) + extra
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if echo:
            sys.stdout.write(line); sys.stdout.flush()
    p.wait(); return p.returncode

def gerar(nome, tenant_id):
    print('='*64); print('CLIENTE:', nome); print('='*64, flush=True)
    t0 = time.time()
    run_ps(['-OnlyTenant', tenant_id])
    novos_html = [h for h in glob.glob(DEST + '/automation/output/**/reports/*.html', recursive=True)
                  if os.path.getmtime(h) >= t0 - 2]
    novos_pptx = [pp for pp in glob.glob(DEST + '/automation/output/**/reports/*.pptx', recursive=True)
                  if os.path.getmtime(pp) >= t0 - 2]
    if not novos_html and not novos_pptx:
        print('\n>>> Nenhum relatório gerado para', nome, '- veja os erros acima.'); return
    for h in novos_html:
        print('  baixando (HTML):', os.path.basename(h), flush=True)
        files.download(h)
    for pp in novos_pptx:
        print('  baixando (PPTX):', os.path.basename(pp), flush=True)
        files.download(pp)
    novos_deck = [d for d in glob.glob(DEST + '/automation/output/**/reports/*.deck.pdf', recursive=True)
                  if os.path.getmtime(d) >= t0 - 2]
    for d in novos_deck:
        print('  baixando (DECK PDF):', os.path.basename(d), flush=True)
        files.download(d)
    print('\n>>> OK -', nome, 'baixado (HTML + PPTX + deck).')

print('Pronto. Vá para a seção 5 e rode a célula do cliente que quiser.')

## 5. Clientes — rode a célula de cada cliente
Cada célula abaixo gera e baixa **um cliente** (HTML + PPTX). Rode a que você precisa; pode repetir e rodar em qualquer ordem. O nome do cliente aparece na própria célula.

### Data center: us-cloud

In [ ]:
gerar('AGILE2 CONSULTING', 'c475fe68-6e4c-42ad-ac35-a8db91d2e9e7')

In [ ]:
gerar('AUXILIUM CORRETORA', '4f26760f-3abb-4bf0-bf28-ee919fb6e890')

In [ ]:
gerar('SOORO', '585bf402-731c-4f4f-bbb6-2c838ec0d6e4')

In [ ]:
gerar('BIOCAL', '599c6865-e26c-427d-8fec-7d9a94924bd5')

In [ ]:
gerar('GRUPO H+ BRASIL', '2d18ca2d-8781-4804-9221-bb066c3b5561')

### Data center: br02-cloud

In [ ]:
gerar('Conselho Regional de Odontologia da Bahia', '1f7dcbee-9901-44b1-bbc0-ccc1fc42ed1e')

In [ ]:
gerar('SAAE MG', '41bba3c0-7770-4dda-ab7a-8c75f539a7be')

In [ ]:
gerar('Coren RS', 'f60dd4a0-cccc-4c9e-b133-1aa2bf6ce9cc')

In [ ]:
gerar('Passebus', 'e24db3bb-63cf-4c8c-abeb-73a94fa046c3')

In [ ]:
gerar('Prefeitura de Taquari', 'c8bc4072-a716-4c22-967d-84e125c3adac')

In [ ]:
gerar('PREFEITURA DE ARAGUAINA', 'c3b3c9c5-0fe9-411b-9f66-d8291fb19a54')

In [ ]:
gerar('Prefeitura Municipal de Itapejara D\'Oeste', 'e92c7bc6-d941-46eb-9122-f05ae7968c6d')

In [ ]:
gerar('FUNPREVAL - GO', 'c622eee9-6e84-4f64-80d7-4f628dfb6b56')

In [ ]:
gerar('UNIMED PLANALTO NORTE', '11b6f8f4-a2b5-4976-9518-354248b8ec32')

In [ ]:
gerar('Camara Campestre', '611b219e-0d63-491e-90ce-5c7fe24fee82')

In [ ]:
gerar('Prefeitura Municipal de Terra Roxa - SP', '30872e42-312f-4cab-9baf-b320bfe7c56f')

In [ ]:
gerar('Prefeitura Municipal de Carnauba dos Dantas', 'b35bd785-c33f-444d-96d2-de471fa0e7bf')

In [ ]:
gerar('Servico Autonomo de Agua e Esgoto de Salto (SAAE-SP)', '3c0c9333-ee22-47a1-9fbc-46f0a9e8f434')

In [ ]:
gerar('CREF14 - GO/TO', '5b7fb0fb-35fc-48d7-af97-45420383e9eb')

In [ ]:
gerar('CORE - DF', 'f0534e41-2ab5-41ce-af58-f2437e00a402')

In [ ]:
gerar('Contego Trial', '0e962b59-6fc3-4d5c-963a-d57d8dbecb40')

In [ ]:
gerar('PREFEITURA DE AIMORES', '3427e892-0333-45b5-acb7-9c990185218f')

In [ ]:
gerar('MERCOS', 'a367ed9b-d7e5-454b-8ad6-4d21bb0b95a6')

In [ ]:
gerar('CAMARA MUNICIPAL DE CANGUCU', '59f899f9-b31e-4714-b203-8e2c20f56dd5')